# Singular AI Model approach

This approach focuses on a single LLM which is multi lingual to use the ` 58 reports ` and give a JSON file as an output which will be validated and saved if the labels are correct or goes back to training and labelling

In [1]:
import asyncio
import ollama
from ollama import chat
from pathlib import Path
import pandas as pd 

print("Import Successful")

Import Successful


# Prepare Report List

In [3]:
IN_DIR = Path("Sample_Data")
raw_data = IN_DIR / "Sample_Data.csv"
print("Data Import Successful")

# Isolate Report 
raw_df = pd.read_csv(raw_data)
reports = raw_df["Report"]

if all(reports):
    print("Report Isolation Successful")
else:
    print("Report Isolation Unsuccessful")

Data Import Successful
Report Isolation Successful


# JSON Creation

In [10]:
import json 
from tqdm import tqdm

results = []

for idx, report in tqdm(
    enumerate(reports),
    total=len(reports)
):
    response = chat(
        model="gemma4:e4b",
        messages=[
            {
                "role": "user",
                "content": prompt.format(report=report)
            }
        ],
        format="json"
    )

    parsed = json.loads(response["message"]["content"])
    results.append({
        "idx": idx,
        "report": report,
        "model": "gemma3:4b",
        "output": parsed
    })

print("Complete")

 84%|████████▍ | 49/58 [26:56<04:56, 32.98s/it]


JSONDecodeError: Unterminated string starting at: line 45 column 9 (char 1430)

In [4]:
OUT_DIR = Path("Sample_Data")
OUT_DIR.mkdir(exist_ok=True)

gemma_responses = pd.json_normalize(results)

gemma_responses = gemma_responses.drop(
    columns=["idx", "model"],
    errors="ignore"
)

gemma_responses.to_json(
    OUT_DIR / "Gemma-Responses.json",
    orient="records",
    indent=2,
    force_ascii=False
)

NameError: name 'results' is not defined

# Validation

The two approaches that are available for validation are:
 * Macro Accuracy -> Accuracy is calculated per class
 * Language Based Accuracy -> Similar to Macro Accuracy but accuracy is based per language  

In [5]:
raw_df = raw_df[~raw_df['ACL'].isnull()]
LABELS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 
          'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker\'s", 
          'Contusion', 'Fracture']
raw_df = raw_df[LABELS]

In [10]:
IN_DIR = Path("Sample_Data")
try:
    chatgpt_responses = pd.read_json(IN_DIR / "ChatGPT-Responses.json")
except Exception as e:
    print(e)

OUTPUT_LABEL = [
    "output.ACL.status", "output.MCL.status", "output.Medial Meniscus.status", "output.Lateral Meniscus.status", "output.Medial OA.status",
    "output.Lateral OA.status", "output.PF OA.status", "output.Effusion.status", "output.Synovitis.status", "output.Baker\'s.status", 
    "output.Contusion.status", "output.Fracture.status"
]

chatgpt_responses = chatgpt_responses[LABELS]
chatgpt_responses.info()

<class 'pandas.DataFrame'>
RangeIndex: 58 entries, 0 to 57
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   ACL               58 non-null     object
 1   MCL               58 non-null     object
 2   Medial Meniscus   58 non-null     object
 3   Lateral Meniscus  58 non-null     object
 4   Medial OA         58 non-null     object
 5   Lateral OA        58 non-null     object
 6   PF OA             58 non-null     object
 7   Effusion          58 non-null     object
 8   Synovitis         58 non-null     object
 9   Baker's           58 non-null     object
 10  Contusion         58 non-null     object
 11  Fracture          58 non-null     object
dtypes: object(12)
memory usage: 5.6+ KB


In [49]:
from sklearn.metrics import balanced_accuracy_score
import numpy as np

def macro_accuracy(true_df : pd.DataFrame, 
                   predict_df : pd.DataFrame) -> float:
    """
    Macro accuracy is calculated by taking the average of the accuracy scores for each class 
    in a multi-class classification problem. This means you compute the accuracy for each 
    class separately and then average those values.

    Args:
        true_df : pd.DataFrame = Dataframe which contains the true value 
        predict_df : pd.DataFrame = DataFrame which was predicted using a model/LLM

        
    Output:
        Accuracy of the model in floating point (double precision)
    """

    macro_balanced_accuracy = np.mean([
        balanced_accuracy_score(true_df[label], predict_df[label])
        for label in OUTPUT_LABEL
    ])  

    return macro_balanced_accuracy

macro_accuracy_score = macro_accuracy(raw_df, gemma_responses)
macro_accuracy_score

KeyError: 'output.ACL.status'

In [50]:
# Self-contained validation: this cell can be run after restarting the kernel.
from pathlib import Path
import re
import numpy as np
import pandas as pd
from sklearn.metrics import balanced_accuracy_score

LABELS = [
    'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus',
    'Medial OA', 'Lateral OA', 'PF OA', 'Effusion',
    'Synovitis', "Baker's", 'Contusion', 'Fracture'
]

data_dir = Path('Sample_Data')
true_df = (
    pd.read_csv(data_dir / 'Sample_Data.csv')
    .dropna(subset=['ACL'])
    .loc[:, LABELS]
    .reset_index(drop=True)
    .astype(int)
)
gemma_df = pd.read_json(data_dir / 'Gemma-Responses.json').reset_index(drop=True)

if len(true_df) != len(gemma_df):
    raise ValueError(
        f'Row mismatch: ground truth has {len(true_df)} rows but Gemma has '
        f'{len(gemma_df)}. Finish/regenerate the missing responses first.'
    )

def normalize_status(value):
    if pd.isna(value):
        return np.nan
    token = re.sub(r'\s+', '', str(value).lower())
    if token in {'1', '1.0', 'present', 'present:1'}:
        return 1
    if token in {
        '0', '0.0', 'absent', 'absent:0',
        'not_mentioned', 'not_mentioned:0', 'uncertain'
    }:
        return 0
    return np.nan

predict_df = pd.DataFrame({
    label: gemma_df[f'output.{label}.status'].map(normalize_status)
    for label in LABELS
})

missing = [
    (int(row), label)
    for label in LABELS
    for row in predict_df.index[predict_df[label].isna()]
]
if missing:
    print(f'Filled {len(missing)} missing/invalid prediction(s) with 0: {missing}')

# The prompt defines uncertain/not-mentioned as 0, so use the same fallback for invalid output.
predict_df = predict_df.fillna(0).astype(int)

per_label_accuracy = pd.Series(
    {
        label: balanced_accuracy_score(true_df[label], predict_df[label])
        for label in LABELS
    },
    name='balanced_accuracy'
)
macro_accuracy_score = float(per_label_accuracy.mean())

print(per_label_accuracy.to_frame().to_string())
print(f'Macro balanced accuracy: {macro_accuracy_score:.4f}')


Filled 1 missing/invalid prediction(s) with 0: [(18, 'Contusion')]
                  balanced_accuracy
ACL                        0.832108
MCL                        0.867347
Medial Meniscus            0.777644
Lateral Meniscus           0.734161
Medial OA                  0.827132
Lateral OA                 0.781431
PF OA                      0.772844
Effusion                   0.652174
Synovitis                  0.676225
Baker's                    0.914855
Contusion                  0.805668
Fracture                   0.838889
Macro balanced accuracy: 0.7900


In [51]:
# Self-contained ChatGPT validation: safe to run after restarting the kernel.
from pathlib import Path
import re
import numpy as np
import pandas as pd
from sklearn.metrics import balanced_accuracy_score

LABELS = [
    'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus',
    'Medial OA', 'Lateral OA', 'PF OA', 'Effusion',
    'Synovitis', "Baker's", 'Contusion', 'Fracture'
]

data_dir = Path('Sample_Data')
true_df = (
    pd.read_csv(data_dir / 'Sample_Data.csv')
    .dropna(subset=['ACL'])
    .loc[:, LABELS]
    .reset_index(drop=True)
    .astype(int)
)
chatgpt_df = pd.read_json(data_dir / 'ChatGPT-Responses.json').reset_index(drop=True)

if len(true_df) != len(chatgpt_df):
    raise ValueError(
        f'Row mismatch: ground truth has {len(true_df)} rows but ChatGPT has '
        f'{len(chatgpt_df)}. Finish/regenerate the missing responses first.'
    )

def normalize_chatgpt_status(value):
    # ChatGPT stores each label as {'status': ..., 'evidence': ...}.
    if isinstance(value, dict):
        value = value.get('status')
    if pd.isna(value):
        return np.nan
    token = re.sub(r'\s+', '', str(value).lower())
    if token in {'1', '1.0', 'present', 'present:1'}:
        return 1
    if token in {
        '0', '0.0', 'absent', 'absent:0',
        'not_mentioned', 'not_mentioned:0', 'uncertain'
    }:
        return 0
    return np.nan

chatgpt_predict_df = pd.DataFrame({
    label: chatgpt_df[label].map(normalize_chatgpt_status)
    for label in LABELS
})

missing = [
    (int(row), label)
    for label in LABELS
    for row in chatgpt_predict_df.index[chatgpt_predict_df[label].isna()]
]
if missing:
    print(f'Filled {len(missing)} missing/invalid prediction(s) with 0: {missing}')

chatgpt_predict_df = chatgpt_predict_df.fillna(0).astype(int)

chatgpt_per_label_accuracy = pd.Series(
    {
        label: balanced_accuracy_score(true_df[label], chatgpt_predict_df[label])
        for label in LABELS
    },
    name='balanced_accuracy'
)
chatgpt_macro_accuracy = float(chatgpt_per_label_accuracy.mean())

print(chatgpt_per_label_accuracy.to_frame().to_string())
print(f'ChatGPT macro balanced accuracy: {chatgpt_macro_accuracy:.4f}')


                  balanced_accuracy
ACL                        0.870098
MCL                        0.862812
Medial Meniscus            0.840144
Lateral Meniscus           0.798758
Medial OA                  0.896899
Lateral OA                 0.823985
PF OA                      0.796654
Effusion                   0.673913
Synovitis                  0.676225
Baker's                    0.903986
Contusion                  0.778003
Fracture                   0.770833
ChatGPT macro balanced accuracy: 0.8077
